In [1]:
import pandas as pd
import numpy as np
import re

# 1. Load original ball-by-ball dataset
df = pd.read_csv("../data/ipl_2026_ball_by_ball.csv")

# Clean column names
df.columns = df.columns.str.strip()


df = df[df["innings"] == 1].copy()

# 2. Convert required columns to numeric

for col in ["wides", "noballs"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["wides"] = df["wides"].fillna(0)
df["noballs"] = df["noballs"].fillna(0)

# 3. Extract runs and wickets from scores such as 33/3
def extract_runs(value):
    if pd.isna(value):
        return np.nan

    match = re.match(r"^\s*(\d+)\s*(?:/|$)", str(value))
    return int(match.group(1)) if match else np.nan


def extract_wickets(value):
    if pd.isna(value):
        return np.nan

    match = re.match(r"^\s*\d+\s*/\s*(\d+)", str(value))
    return int(match.group(1)) if match else np.nan


df["Score_Runs"] = df["team_score"].apply(extract_runs)
df["Score_Wickets"] = df["team_score"].apply(extract_wickets)

# 4. Check that score values were extracted correctly
if df["Score_Runs"].isna().any():
    raise ValueError(
        "Some team_score values could not be parsed. "
        "Inspect df['team_score'].dropna().head(20)."
    )

# 5. Preserve original delivery order within each innings
df["_original_order"] = np.arange(len(df))

df = df.sort_values(
    ["match_id", "innings", "_original_order"],
    kind="stable"
).copy()

# 6. Identify legal deliveries
df["legal_ball"] = (
    (df["wides"] == 0) &
    (df["noballs"] == 0)
)

df["Legal_Balls"] = (
    df.groupby(["match_id", "innings"])["legal_ball"]
    .cumsum()
)

# 7. Get final score from the last delivery of each innings
# IMPROVED: also keep wickets and legal balls at the last delivery so we can
# tell whether the innings was really completed
final_scores = (
    df.groupby(["match_id", "innings"], sort=False)
    .tail(1)[["match_id", "innings", "Score_Runs", "Score_Wickets", "Legal_Balls"]]
    .rename(columns={
        "Score_Runs": "Final_Runs",
        "Score_Wickets": "Final_Wickets",
        "Legal_Balls": "Final_Legal_Balls",
    })
)

df = df.merge(
    final_scores,
    on=["match_id", "innings"],
    how="left"
)

# IMPROVED: drop shortened / abandoned innings. A complete innings is either
# all out (10 wickets) or has used all 20 overs (120 legal balls).
complete = (df["Final_Wickets"] == 10) | (df["Final_Legal_Balls"] >= 120)
n_total = df["match_id"].nunique()
df = df[complete].copy()
print(f"Kept {df['match_id'].nunique()} of {n_total} first innings "
      f"(dropped {n_total - df['match_id'].nunique()} shortened/abandoned)")

# 8. Generate snapshots at 5, 10 and 15 overs
snapshots = []

for (match_id, innings), group in df.groupby(
    ["match_id", "innings"], sort=False
):

    for overs in [5, 10, 15]:

        required_balls = overs * 6

        # Select the delivery where the legal-ball count first reaches this
        # checkpoint.
        # IMPROVED: use the FIRST such row (iloc[0]), not the last. Rows after
        # it with the same count are wides/no-balls from the NEXT over, which
        # would add runs that belong after the checkpoint.
        checkpoint = group[
            group["Legal_Balls"] == required_balls
        ]

        if checkpoint.empty:
            continue

        record = checkpoint.iloc[0]

        snapshots.append({
            "Match_ID": match_id,
            "Innings": innings,
            "Overs": overs,
            "Current_Runs": int(record["Score_Runs"]),
            "Wickets": int(record["Score_Wickets"]),
            "Final_Runs": int(record["Final_Runs"])
        })

# 9. Create the final prediction dataset
model_df = pd.DataFrame(snapshots)

# 10. Validate the dataset
model_df = model_df.dropna()

# IMPROVED: the old checks (current <= final, wickets 0-10) are always true
# for cumulative scores, so they removed nothing. A useful check is to drop
# snapshots where the team is already all out: the score is then already final.
model_df = model_df[model_df["Wickets"] < 10].copy()

# 11. Save the new CSV
model_df.to_csv(
    "../data/ipl_score_prediction.csv",
    index=False
)

# 12. Display results
print("NEW DATASET CREATED SUCCESSFULLY")
print("--------------------------------")

print("\nDataset shape:", model_df.shape)

print("\nRows per checkpoint:")
print(model_df["Overs"].value_counts().sort_index())

print("\nFirst 15 rows:")
print(model_df.head(15).to_string(index=False))

print("\nData types:")
print(model_df.dtypes)

print("\nMissing values:")
print(model_df.isnull().sum())

print("\nSaved to: data/ipl_score_prediction.csv")

Kept 71 of 74 first innings (dropped 3 shortened/abandoned)
NEW DATASET CREATED SUCCESSFULLY
--------------------------------

Dataset shape: (213, 6)

Rows per checkpoint:
Overs
5     71
10    71
15    71
Name: count, dtype: int64

First 15 rows:
 Match_ID  Innings  Overs  Current_Runs  Wickets  Final_Runs
        1        1      5            33        3         201
        1        1     10            87        3         201
        1        1     15           142        5         201
        2        1      5            68        0         220
        2        1     10           120        2         220
        2        1     15           167        3         220
        3        1      5            36        3         127
        3        1     10            70        6         127
        3        1     15            93        8         127
        4        1      5            47        1         162
        4        1     10            84        2         162
        4        1  